# CBCT ToothSeg label-all teacher dataset

Objective: run the ToothSeg teacher over every raw CBCT case and persist
standardized image/label pairs in Drive for browser-model training.

This notebook is resumable. It skips cases that already have a saved label unless
`SKIP_EXISTING = False`.

Expected final dataset:

`cbct-notebook/clinic-raw/Dataset_clinic/imagesTr/{case}_0000.nii.gz`

`cbct-notebook/clinic-raw/Dataset_clinic/labelsTr/{case}.nii.gz`

Runtime: T4 GPU or better.

## 1. Setup

In [ ]:
!nvidia-smi -L || true

In [ ]:
!test -d /content/ToothSeg || git clone -q https://github.com/MIC-DKFZ/ToothSeg.git /content/ToothSeg
!pip -q install -e /content/ToothSeg SimpleITK scipy pandas pydicom nibabel 2>&1 | tail -8
import nnunetv2, torch
print('nnunetv2 OK; cuda:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
import csv, json, os, re, shutil, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk

PROJECT_CANDIDATES = [
    Path('/content/drive/MyDrive/Projects/Health/CBCT'),
    Path('/content/drive/MyDrive/Projects/Health/cbct'),
]
PROJECT = next((p for p in PROJECT_CANDIDATES if p.exists()), PROJECT_CANDIDATES[0])
WORK = PROJECT / 'cbct-notebook'
OUT = PROJECT / 'cbct-outputs'
DATASET = WORK / 'clinic-raw' / 'Dataset_clinic'
IMAGE_DIR = DATASET / 'imagesTr'
LABEL_DIR = DATASET / 'labelsTr'
RUN_OUT = OUT / 'toothseg-label-all'
RECOVER = WORK / 'recover_toothseg_semantic_labels.py'

# Put raw NIfTI/DICOM case folders in one of these folders, or add explicit
# entries to MANUAL_CASES below.
RAW_ROOTS = [
    WORK / 'raw-cases',
    WORK / 'raw',
    WORK / 'dicom',
    WORK / 'DICOM',
    PROJECT / 'raw-cases',
    PROJECT / 'raw',
    PROJECT / 'dicom',
    PROJECT / 'DICOM',
]

# Example:
# MANUAL_CASES = [{'case': 'case_001', 'path': '/content/drive/MyDrive/.../case_001_dicom'}]
MANUAL_CASES = []

CHECKPOINT_CANDIDATES = [
    PROJECT / 'checkpoints' / 'nnUNet_results' / 'ToothSeg',
    WORK / 'checkpoints' / 'nnUNet_results' / 'ToothSeg',
    Path('/content/drive/MyDrive/UniFi Drive_UNAS Pro 8/UNAS Pro 8_Main Backup/Main/cbct/checkpoints/nnUNet_results/ToothSeg'),
]
nnunet_results = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
assert nnunet_results is not None, 'Missing ToothSeg checkpoints. Add path to CHECKPOINT_CANDIDATES.'

os.environ['nnUNet_results'] = str(nnunet_results)
os.environ['nnUNet_raw'] = '/content/nnraw'
os.environ['nnUNet_preprocessed'] = '/content/nnprep'
os.environ['nnUNet_compile'] = 'F'
for path in [WORK, OUT, WORK / 'raw-cases', IMAGE_DIR, LABEL_DIR, RUN_OUT, Path('/content/nnraw'), Path('/content/nnprep')]:
    path.mkdir(parents=True, exist_ok=True)

TS = Path('/content/ToothSeg/toothseg/toothseg')
assert TS.exists(), TS

SKIP_EXISTING = True
MAX_CASES = 0      # 0 = all pending cases; set 1 for a smoke test
BATCH_SIZE = 2     # ToothSeg is heavy; keep this small on T4
SAVE_INTERMEDIATES = True

print('PROJECT:', PROJECT)
print('WORK:', WORK)
print('DATASET:', DATASET)
print('RUN_OUT:', RUN_OUT)
print('nnUNet_results:', nnunet_results)
print('raw roots that exist:', [str(p) for p in RAW_ROOTS if p.exists()])

## 3. Discover raw cases

Supported inputs: NIfTI/MHA image files or DICOM folders. The notebook only scans
the configured raw folders, not all of MyDrive.

In [ ]:
IMAGE_EXTS = ('.nii', '.nii.gz', '.mha', '.mhd')
LABEL_HINTS = ('label', 'labels', 'seg', 'mask', 'prediction', 'toothseg', 'final')

def strip_medical_suffix(path: Path) -> str:
    name = path.name
    for suffix in ['.nii.gz', '.nii', '.mha', '.mhd']:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    if name.endswith('_0000'):
        name = name[:-5]
    return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()

def is_probably_label(path: Path) -> bool:
    lower = path.name.lower()
    return any(hint in lower for hint in LABEL_HINTS) and '_0000' not in lower

def has_dicom_series(path: Path) -> bool:
    try:
        return bool(sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(path)))
    except Exception:
        return False

def discover_raw_cases() -> list[dict]:
    cases: dict[str, dict] = {}
    for item in MANUAL_CASES:
        case_id = re.sub(r'[^A-Za-z0-9]+', '_', item['case']).strip('_').lower()
        cases[case_id] = {'case': case_id, 'path': item['path'], 'kind': 'manual'}

    for root in RAW_ROOTS:
        if not root.exists():
            continue
        for path in sorted(root.rglob('*')):
            if not path.exists() or path.name.startswith('.'):
                continue
            if path.is_file() and path.name.lower().endswith(IMAGE_EXTS) and not is_probably_label(path):
                case_id = strip_medical_suffix(path)
                cases.setdefault(case_id, {'case': case_id, 'path': str(path), 'kind': 'image'})
            elif path.is_dir():
                # Only spend DICOM detection work on dirs that actually contain files.
                try:
                    has_files = any(child.is_file() and not child.name.startswith('.') for child in path.iterdir())
                except Exception:
                    has_files = False
                if has_files and has_dicom_series(path):
                    case_id = re.sub(r'[^A-Za-z0-9]+', '_', path.name).strip('_').lower()
                    cases.setdefault(case_id, {'case': case_id, 'path': str(path), 'kind': 'dicom'})
    return [cases[k] for k in sorted(cases)]

raw_cases = discover_raw_cases()
existing_images = []
for image in sorted(IMAGE_DIR.glob('*_0000.nii*')):
    case_id = strip_medical_suffix(image)
    existing_images.append({'case': case_id, 'path': str(image), 'kind': 'standardized'})

cases_by_id = {item['case']: item for item in existing_images}
for item in raw_cases:
    cases_by_id[item['case']] = item
cases = [cases_by_id[k] for k in sorted(cases_by_id)]

if not cases:
    print('No raw cases found. Put raw CBCTs under one of RAW_ROOTS or fill MANUAL_CASES.')
assert cases, 'No input cases found.'
display(pd.DataFrame(cases))

## 4. Standardize images into Dataset_clinic/imagesTr

In [ ]:
def read_largest_dicom_series(folder: Path) -> sitk.Image:
    series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(folder))
    if not series_ids:
        raise RuntimeError(f'No DICOM series in {folder}')
    best_files = []
    for series_id in series_ids:
        files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(folder), series_id)
        if len(files) > len(best_files):
            best_files = list(files)
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(best_files)
    return reader.Execute()

def standardize_case(item: dict) -> dict:
    case_id = item['case']
    image_out = IMAGE_DIR / f'{case_id}_0000.nii.gz'
    label_out = LABEL_DIR / f'{case_id}.nii.gz'
    if image_out.exists():
        return {**item, 'image': str(image_out), 'label': str(label_out), 'standardized': 'exists'}
    src = Path(item['path'])
    print('STANDARDIZE', case_id, 'from', src)
    if item['kind'] == 'dicom' or src.is_dir():
        image = read_largest_dicom_series(src)
    else:
        image = sitk.ReadImage(str(src))
    sitk.WriteImage(image, str(image_out), True)
    return {**item, 'image': str(image_out), 'label': str(label_out), 'standardized': 'written'}

standardized = [standardize_case(item) for item in cases]
df = pd.DataFrame(standardized)
df['labelExists'] = df['label'].map(lambda p: Path(p).exists())
df.to_csv(RUN_OUT / 'case-inventory.csv', index=False)
display(df)
print('inventory:', RUN_OUT / 'case-inventory.csv')

## 5. Run ToothSeg for missing labels

In [ ]:
def run_cmd(args: list[str | Path]) -> None:
    args = [str(arg) for arg in args]
    print('>>', ' '.join(args), flush=True)
    subprocess.run(args, check=True)

def chunks(items: list[dict], size: int):
    for start in range(0, len(items), max(1, size)):
        yield start // max(1, size), items[start:start + max(1, size)]

def label_summary(path: Path) -> dict:
    image = sitk.ReadImage(str(path))
    data = sitk.GetArrayFromImage(image).astype(np.int32, copy=False)
    labels, counts = np.unique(data, return_counts=True)
    rows = [
        {'label': int(label), 'voxels': int(count)}
        for label, count in zip(labels.tolist(), counts.tolist())
        if int(label) != 0
    ]
    return {
        'labelCount': len(rows),
        'positiveVoxels': int(sum(row['voxels'] for row in rows)),
        'labels': rows,
    }

pending = []
for item in standardized:
    label = Path(item['label'])
    if SKIP_EXISTING and label.exists():
        continue
    pending.append(item)
if MAX_CASES:
    pending = pending[:MAX_CASES]

print('pending cases:', [item['case'] for item in pending])
if not pending:
    print('No pending cases. All discovered cases already have labels.')

manifest_rows = []
for batch_index, batch in chunks(pending, BATCH_SIZE):
    W = Path('/content/toothseg-label-all') / f'batch_{batch_index:03d}'
    shutil.rmtree(W, ignore_errors=True)
    input_dir = W / 'input' / 'imagesTs'
    input_dir.mkdir(parents=True, exist_ok=True)
    for item in batch:
        shutil.copy2(item['image'], input_dir / f"{item['case']}_0000.nii.gz")

    run_cmd(['python', TS / 'test_set_prediction_and_eval' / 'resize_test_set.py',
             '-i', input_dir, '-o', W / 'input' / 'imagesTs_resized_02'])
    run_cmd(['nnUNetv2_predict', '--disable_tta',
             '-i', input_dir, '-o', W / 'semseg',
             '-d', 'Dataset121_ToothFairy2_Teeth',
             '-tr', 'nnUNetTrainer_onlyMirror01_DASegOrd0',
             '-p', 'nnUNetPlans',
             '-c', '3d_fullres_resample_torch_256_bs8_ctnorm',
             '-f', '5', '-device', 'cuda'])
    run_cmd(['nnUNetv2_predict', '--disable_tta',
             '-i', W / 'input' / 'imagesTs_resized_02', '-o', W / 'instseg',
             '-d', 'Dataset123_ToothFairy2fixed_teeth_spacing02_brd3px',
             '-tr', 'nnUNetTrainer', '-p', 'nnUNetPlans',
             '-c', '3d_fullres_resample_torch_192_bs8_ctnorm',
             '-f', '5', '-device', 'cuda'])
    run_cmd(['python', TS / 'postprocess_predictions' / 'border_core_to_instances.py',
             '-i', W / 'instseg', '-o', W / 'inst', '-np', '2'])
    run_cmd(['python', TS / 'postprocess_predictions' / 'resize_predictions.py',
             '-i', W / 'inst', '-o', W / 'inst_resized',
             '-ref', input_dir, '-np', '2'])
    run_cmd(['python', TS / 'postprocess_predictions' / 'assign_majority_tooth_labels.py',
             '-ifolder', W / 'inst_resized', '-sfolder', W / 'semseg',
             '-o', W / 'final', '-np', '2'])

    for item in batch:
        case_id = item['case']
        label_dst = Path(item['label'])
        final = W / 'final' / f'{case_id}.nii.gz'
        if not final.exists():
            matches = sorted((W / 'final').glob(f'{case_id}*.nii.gz'))
            if not matches:
                raise FileNotFoundError(f'Missing final ToothSeg output for {case_id}')
            final = matches[0]
        semseg = W / 'semseg' / f'{case_id}.nii.gz'

        final_archive = RUN_OUT / 'final_prediction' / f'{case_id}.nii.gz'
        final_archive.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(final, final_archive)

        if SAVE_INTERMEDIATES and semseg.exists():
            semseg_archive = RUN_OUT / 'semseg' / f'{case_id}.nii.gz'
            semseg_archive.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(semseg, semseg_archive)

        label_source = final_archive
        recovered_archive = RUN_OUT / 'final_prediction_recovered' / f'{case_id}.nii.gz'
        if RECOVER.exists() and semseg.exists():
            run_cmd(['python', RECOVER, '--semantic', semseg, '--final', final,
                     '--output', recovered_archive, '--min-voxels', '10000',
                     '--min-component-voxels', '500'])
            label_source = recovered_archive

        shutil.copy2(label_source, label_dst)
        summary = label_summary(label_dst)
        manifest_rows.append({
            'case': case_id,
            'status': 'ok',
            'image': item['image'],
            'label': str(label_dst),
            'labelCount': summary['labelCount'],
            'positiveVoxels': summary['positiveVoxels'],
        })
        (RUN_OUT / 'label-summaries').mkdir(parents=True, exist_ok=True)
        (RUN_OUT / 'label-summaries' / f'{case_id}.json').write_text(json.dumps(summary, indent=2))
        print('SAVED', case_id, 'labels:', summary['labelCount'], 'voxels:', summary['positiveVoxels'])

manifest_path = RUN_OUT / 'teacher-label-manifest.csv'
existing_rows = []
if manifest_path.exists():
    existing_rows = list(csv.DictReader(manifest_path.open()))
all_rows = existing_rows + manifest_rows
if all_rows:
    with manifest_path.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['case', 'status', 'image', 'label', 'labelCount', 'positiveVoxels'])
        writer.writeheader()
        writer.writerows(all_rows)
print('manifest:', manifest_path)

## 6. Verify the persisted dataset

In [ ]:
pairs = []
for image in sorted(IMAGE_DIR.glob('*_0000.nii*')):
    case_id = strip_medical_suffix(image)
    label = LABEL_DIR / f'{case_id}.nii.gz'
    pairs.append({
        'case': case_id,
        'image': str(image),
        'label': str(label),
        'labelExists': label.exists(),
    })
df = pd.DataFrame(pairs)
display(df)
print('image count:', len(df))
print('labeled count:', int(df['labelExists'].sum()) if len(df) else 0)
assert len(df) > 0, 'No standardized images were written.'
assert df['labelExists'].all(), 'Some images still do not have labels.'

## 7. Next step

After this notebook finishes, rerun `cbct_toothseg_multi_teacher_colab.ipynb`.
It will consume the persisted `imagesTr`/`labelsTr` pairs and train/evaluate the
browser-sized YOLO student on the full teacher-labeled dataset.